# Preparation of the test files

In this Jupyter Notbook all necessary test files are loaded and prepared. This means the PointCloud datafiles as well as the necessary data to define the borders is imported and the laz-files cut so they only include the the areas on which the model will be tested on

In [55]:
import geopandas as gpd
from geopandas import clip
import pandas as pd
import matplotlib.pyplot as plt
import folium

zurich = gpd.read_file("C:/Users/flori/Desktop/pa-1_data/richtplan/richtplan_zürich/Kantonaler_Richtplan_-OGD/Kantonaler_Richtplan_-OGD.gpkg", layer= "KANT_RICHT_F")

display(zurich.head(3))
print(zurich.shape)
print(zurich['OBJEKTART_NAME'].unique())
print(zurich['UNTERKAPITEL_NAME'].unique())

,OBJEKTART_ID,OBJEKTART_NAME,OBJEKTVERW,KTZH,KAPITEL_NAME,KAPITEL_ABKUERZUNG,UNTERKAPITEL_NAME,UNTERKAPITEL_NUMMER,NR_TABELLE,NAME_TABELLE,...,OBJEKTE_TEXT4,OBJEKTE_TEXT5,TITEL_TEXT1,TITEL_TEXT2,TITEL_TEXT3,TITEL_TEXT4,TITEL_TEXT5,SHAPE_AREA,SHAPE_LEN,geometry
0,202.0,Übriges Landwirtschaftsgebiet,0.0,1.0,LANDSCHAFT,L,Landwirtschaftsgebiet,3.2,,,...,,,,,,,,241.919698,88.601726,"POLYGON ((2677620.518 1249253.503, 2677610.85 ..."
1,202.0,Übriges Landwirtschaftsgebiet,0.0,1.0,LANDSCHAFT,L,Landwirtschaftsgebiet,3.2,,,...,,,,,,,,25609.529081,1217.686120,"POLYGON ((2677809.989 1246417.882, 2677814.387..."
2,202.0,Übriges Landwirtschaftsgebiet,0.0,1.0,LANDSCHAFT,L,Landwirtschaftsgebiet,3.2,,,...,,,,,,,,177.855000,138.410782,"POLYGON ((2678046.284 1247346.471, 2678042.074..."


(671, 24)
['Übriges Landwirtschaftsgebiet' 'Wald' 'Naturschutzgebiet'
 'Fruchtfolgefläche im Landwirtschaftsgebiet' 'Erholungsgebiet' 'Gewässer'
 'Siedlungsgebiet' 'Grundwasserschutzgebiet' 'Zentrumsgebiet'
 'Landschaftsverbindung geplant' 'Naturschutzgebiet (in Gewässern)'
 'Schutzwürdiges Ortsbild' 'Fachübergreifende Gebietsplanungen bestehend'
 'Gewässerrevitalisierung' 'Landschaftsschutzgebiet' 'Freihaltegebiet'
 'Landschaftsförderungsgebiet' 'Bundesinventare (Summe BLN/ML/AG)'
 'Landschaftsverbindung bestehend']
['Landwirtschaftsgebiet' 'Wald' 'Naturschutz' 'Erholung' 'Gewässer' ''
 'Zentrumsgebiet' 'Landschaftsverbindung' 'Schutzwürdiges Ortsbild'
 'Gesamtstrategie' 'Landschaftsschutzgebiet' 'Freihaltegebiet'
 'Landschaftsförderungsgebiet']


In [56]:
# Filter for specific OBJEKTART_NAME values
selected_names = ["Siedlungsgebiet", "Schutzwürdiges Ortsbild"]
filtered = zurich[zurich["OBJEKTART_NAME"].isin(selected_names)]
display(filtered.head(3))
display(filtered.shape)
# Reproject to WGS84 for web map compatibility
filtered_wgs = filtered.to_crs(epsg=4326)

# Center the map on the filtered geometries
centroid = filtered_wgs.geometry.union_all().centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=12)

# Add GeoJSON overlay
folium.GeoJson(filtered_wgs, name="Filtered Zones").add_to(m)

# Show layer control
folium.LayerControl().add_to(m)

m

,OBJEKTART_ID,OBJEKTART_NAME,OBJEKTVERW,KTZH,KAPITEL_NAME,KAPITEL_ABKUERZUNG,UNTERKAPITEL_NAME,UNTERKAPITEL_NUMMER,NR_TABELLE,NAME_TABELLE,...,OBJEKTE_TEXT4,OBJEKTE_TEXT5,TITEL_TEXT1,TITEL_TEXT2,TITEL_TEXT3,TITEL_TEXT4,TITEL_TEXT5,SHAPE_AREA,SHAPE_LEN,geometry
51,101.0,Siedlungsgebiet,0.0,1.0,SIEDLUNG,S,,,,,...,,,,,,,,39572.924719,1191.452851,"POLYGON ((2683270.96 1248235.273, 2683298.067 ..."
81,101.0,Siedlungsgebiet,0.0,1.0,SIEDLUNG,S,,,,,...,,,,,,,,693598.635624,6576.933631,"POLYGON ((2681695.121 1243395.613, 2681720.25 ..."
136,103.0,Schutzwürdiges Ortsbild,20401.0,1.0,SIEDLUNG,S,Schutzwürdiges Ortsbild,2.4,204.1,Schutzwürdiges Ortsbild,...,,,Ortsbild,Koordinationshinweise,,,,989898.821282,7668.042020,"POLYGON ((2683433.521 1247863.913, 2683417.705..."


(15, 24)

In [74]:
gmd_borders = gpd.read_file("C:/Users/flori/Desktop/pa-1_data/gemeindegrenzen/gemeindegrenzen.gpkg", layer = "UP_GEMEINDEN_F")
display(gmd_borders.head(3))
print(gmd_borders.shape)

zurich_borders = gmd_borders[gmd_borders["GEMEINDENAME"] == "Zürich"]
# Slightly shrink the Zurich boundary (e.g. 1 meter inward)
zurich_borders_shrunk = zurich_borders.copy()
zurich_borders_shrunk["geometry"] = zurich_borders_shrunk.geometry.buffer(-5)

,BFS,ARPS,BEZIRKSNAME,ART_TEXT,GEMEINDENAME,ART_CODE,AREA_ROUND,SHAPE_AREA,SHAPE_LEN,geometry
0,154,43,Meilen,Gemeinde,Küsnacht (ZH),1,12364519.0,1.236452e+07,24563.719929,"POLYGON ((2687867.052 1243420.865, 2687888.606..."
1,173,81,Pfäffikon,Gemeinde,Hittnau,1,12955399.0,1.295540e+07,20612.489289,"POLYGON ((2705484.771 1249538.112, 2705487.581..."
2,182,92,Pfäffikon,Gemeinde,Wildberg,1,10566255.0,1.056625e+07,25557.190718,"POLYGON ((2705710.443 1255358.355, 2705723.143..."


(165, 10)


In [75]:
# Ensure both are in the same CRS
if filtered.crs != zurich_borders_shrunk.crs:
    zurich_borders_shrunk = zurich_borders_shrunk.to_crs(filtered.crs)

# Clip the filtered zoning areas to Zürich municipality boundary
filtered_zurich = clip(filtered, zurich_borders_shrunk)

# Reproject to WGS84 for display
filtered_zurich_wgs = filtered_zurich.to_crs(epsg=4326)

# Display in Folium
centroid = filtered_zurich_wgs.geometry.union_all().centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=13)
folium.GeoJson(filtered_zurich_wgs, name="Zürich Zoning (Clipped)").add_to(m)
folium.LayerControl().add_to(m)
m

### Function

In [76]:
import geopandas as gpd
from geopandas import clip
import folium

def load_municipality_boundary(gmd_file: str, municipality_name: str) -> gpd.GeoDataFrame:
    """Load and return the boundary polygon for a given municipality."""
    gmd_borders = gpd.read_file(gmd_file, layer="UP_GEMEINDEN_F")
    boundary = gmd_borders[gmd_borders["GEMEINDENAME"] == municipality_name].copy()
    return boundary

def load_and_filter_zoning(zoning_file: str, selected_names=None) -> gpd.GeoDataFrame:
    """Load Richtplan data and filter by selected OBJEKTART_NAME values."""
    if selected_names is None:
        selected_names = ["Siedlungsgebiet", "Schutzwürdiges Ortsbild"]

    zoning = gpd.read_file(zoning_file, layer="KANT_RICHT_F")
    return zoning[zoning["OBJEKTART_NAME"].isin(selected_names)].copy()

def clip_zoning_to_municipality(zoning_gdf: gpd.GeoDataFrame, muni_boundary: gpd.GeoDataFrame, buffer_shrink=5) -> gpd.GeoDataFrame:
    """Clip zoning polygons to the buffered municipality boundary."""
    if zoning_gdf.crs != muni_boundary.crs:
        muni_boundary = muni_boundary.to_crs(zoning_gdf.crs)

    # Shrink boundary slightly to remove edge slivers
    muni_boundary["geometry"] = muni_boundary.geometry.buffer(-buffer_shrink)
    
    return clip(zoning_gdf, muni_boundary)

def show_on_map(gdf: gpd.GeoDataFrame, zoom=13):
    """Display GeoDataFrame on a Folium map."""
    gdf_wgs = gdf.to_crs(epsg=4326)
    centroid = gdf_wgs.geometry.union_all().centroid
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=zoom)
    folium.GeoJson(gdf_wgs, name="Clipped Zoning").add_to(m)
    folium.LayerControl().add_to(m)
    return m

def process_municipality(zoning_file: str, gmd_file: str, municipality_name: str, show_map=True) -> gpd.GeoDataFrame:
    """
    Full processing pipeline:
    - Load and filter zoning data.
    - Load municipal boundary.
    - Clip zoning to boundary.
    - Optionally preview on map.
    - Return GeoDataFrame for further processing.
    """
    zoning = load_and_filter_zoning(zoning_file)
    muni_boundary = load_municipality_boundary(gmd_file, municipality_name)
    clipped = clip_zoning_to_municipality(zoning, muni_boundary)

    if show_map:
        display(show_on_map(clipped))

    return clipped


#### Test

In [81]:
richtplan_path = "C:/Users/flori/Desktop/pa-1_data/richtplan/winterthur/Kantonaler_Richtplan_-OGD/Kantonaler_Richtplan_-OGD.gpkg"
gmd_path = "C:/Users/flori/Desktop/pa-1_data/gemeindegrenzen/gemeindegrenzen.gpkg"
municipality = "Winterthur"

zoning_filtered = process_municipality(richtplan_path, gmd_path, municipality)
display(zoning_filtered.head(3))
print(zoning_filtered.shape)


,OBJEKTART_ID,OBJEKTART_NAME,OBJEKTVERW,KTZH,KAPITEL_NAME,KAPITEL_ABKUERZUNG,UNTERKAPITEL_NAME,UNTERKAPITEL_NUMMER,NR_TABELLE,NAME_TABELLE,...,OBJEKTE_TEXT4,OBJEKTE_TEXT5,TITEL_TEXT1,TITEL_TEXT2,TITEL_TEXT3,TITEL_TEXT4,TITEL_TEXT5,SHAPE_AREA,SHAPE_LEN,geometry
242,101.0,Siedlungsgebiet,0.0,1.0,SIEDLUNG,S,,,,,...,,,,,,,,625962.451980,6190.501933,"MULTIPOLYGON (((2701247.065 1257734.234, 27012..."
579,101.0,Siedlungsgebiet,0.0,1.0,SIEDLUNG,S,,,,,...,,,,,,,,245465.953637,2289.271965,"POLYGON ((2701177.7 1258174.237, 2701180.27 12..."
60,101.0,Siedlungsgebiet,0.0,1.0,SIEDLUNG,S,,,,,...,,,,,,,,159012.256988,1969.486206,"POLYGON ((2699527.216 1257745.519, 2699521.029..."


(12, 24)
